# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ayazahmedbscsf23-art/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [1]:
import pandas as pd, numpy as np
from pathlib import Path

# ---- CONFIG: change these to match your data ----
DATA_PATH = Path("work/data/w04_input.csv")   # your input table
OUT_PATH  = Path("work/outputs/baseline_action_score.csv")
ID_COL    = "item_id"
COLS = dict(impressions="impressions", clicks="clicks", position="avg_position",
            conv="conversions", prev_clicks="prev_clicks")   # prev = EARLIER window only
FORBIDDEN = ["product_flag", "is_product", "future", "next_", "label", "target"]
# --------------------------------------------------

REASONS = {
    "R1_LOW_CTR_HIGH_IMPR":  ("Rewrite title / snippet", "Many impressions, CTR below median"),
    "R2_NEAR_PAGE_ONE":      ("Strengthen content / links", "Average position 8-20 with solid impressions"),
    "R3_DECLINING":          ("Investigate drop", "Clicks fell >30% vs earlier window"),
    "R4_CONVERTING_SCALE":   ("Expand / scale", "Good conversion rate, room to grow impressions"),
    "R5_NO_ACTION":          ("Monitor", "No rule fired"),
}
print(pd.DataFrame(REASONS, index=["action", "meaning"]).T.to_string())

def load():
    if DATA_PATH.exists():
        return pd.read_csv(DATA_PATH)
    rng = np.random.default_rng(7)          # DEMO DATA ONLY
    n = 300
    imp = rng.integers(50, 20000, n)
    return pd.DataFrame({ID_COL: range(n), "impressions": imp,
        "clicks": (imp * rng.uniform(0.002, 0.08, n)).astype(int),
        "avg_position": rng.uniform(1, 40, n).round(1),
        "conversions": rng.integers(0, 40, n),
        "prev_clicks": (imp * rng.uniform(0.002, 0.08, n)).astype(int)})


                                          action                                         meaning
R1_LOW_CTR_HIGH_IMPR     Rewrite title / snippet              Many impressions, CTR below median
R2_NEAR_PAGE_ONE      Strengthen content / links    Average position 8-20 with solid impressions
R3_DECLINING                    Investigate drop              Clicks fell >30% vs earlier window
R4_CONVERTING_SCALE               Expand / scale  Good conversion rate, room to grow impressions
R5_NO_ACTION                             Monitor                                   No rule fired


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [2]:
df = load()
c = COLS
df["ctr"]      = df[c["clicks"]] / df[c["impressions"]].clip(lower=1)
df["cvr"]      = df[c["conv"]] / df[c["clicks"]].clip(lower=1)
df["click_chg"] = (df[c["clicks"]] - df[c["prev_clicks"]]) / df[c["prev_clicks"]].clip(lower=1)

med_ctr   = df["ctr"].median()
big_impr  = df[c["impressions"]].quantile(0.75)
good_cvr  = df["cvr"].quantile(0.75)

# each rule: boolean mask + a 0-1 strength
rules = {
    "R1_LOW_CTR_HIGH_IMPR": (df[c["impressions"]] >= big_impr) & (df["ctr"] < med_ctr),
    "R2_NEAR_PAGE_ONE":     df[c["position"]].between(8, 20) & (df[c["impressions"]] >= df[c["impressions"]].median()),
    "R3_DECLINING":         df["click_chg"] < -0.30,
    "R4_CONVERTING_SCALE":  (df["cvr"] >= good_cvr) & (df[c["clicks"]] > 0),
}
weights = {"R3_DECLINING": 1.0, "R1_LOW_CTR_HIGH_IMPR": 0.9,
           "R2_NEAR_PAGE_ONE": 0.8, "R4_CONVERTING_SCALE": 0.7}

impr_scale = np.log1p(df[c["impressions"]]) / np.log1p(df[c["impressions"]]).max()
scores = pd.DataFrame({k: m.astype(float) * weights[k] * impr_scale for k, m in rules.items()})

df["score"]       = scores.max(axis=1).round(4)
df["reason_code"] = scores.idxmax(axis=1).where(df["score"] > 0, "R5_NO_ACTION")
df["action"]      = df["reason_code"].map(lambda r: REASONS[r][0])
df["rules_fired"] = rules_fired = pd.DataFrame(rules).sum(axis=1)

ranked = df.sort_values("score", ascending=False).reset_index(drop=True)
ranked["rank"] = ranked.index + 1

OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
ranked.to_csv(OUT_PATH, index=False)
print(f"Wrote {len(ranked)} rows -> {OUT_PATH}")
print(ranked["reason_code"].value_counts())

Wrote 300 rows -> work/outputs/baseline_action_score.csv
reason_code
R5_NO_ACTION            129
R3_DECLINING             95
R4_CONVERTING_SCALE      31
R2_NEAR_PAGE_ONE         30
R1_LOW_CTR_HIGH_IMPR     15
Name: count, dtype: int64


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [3]:
WRONG_IF = {
    "R1_LOW_CTR_HIGH_IMPR": "Impressions are brand/navigational noise, or CTR is normal for this position.",
    "R2_NEAR_PAGE_ONE":     "Position is an average over very different queries; real rank is unstable.",
    "R3_DECLINING":         "Seasonality or a tracking gap, not a real loss.",
    "R4_CONVERTING_SCALE":  "Few clicks, so the conversion rate is luck; small sample.",
    "R5_NO_ACTION":         "A real issue hidden by a rule threshold.",
}

def confidence(row):
    n = row[c["clicks"]]
    return "higher (n>=100 clicks)" if n >= 100 else "medium (30-99 clicks)" if n >= 30 else "low (<30 clicks)"

top20 = ranked.head(20).copy()
top20["confidence_note"] = top20.apply(confidence, axis=1)
top20["what_would_make_it_wrong"] = top20["reason_code"].map(WRONG_IF)
show = ["rank", ID_COL, "action", "reason_code", "score", "confidence_note", "what_would_make_it_wrong"]
top20[show]

,rank,item_id,action,reason_code,score,confidence_note,what_would_make_it_wrong
0,1,224,Investigate drop,R3_DECLINING,1.0000,higher (n>=100 clicks),"Seasonality or a tracking gap, not a real loss."
1,2,39,Investigate drop,R3_DECLINING,0.9990,higher (n>=100 clicks),"Seasonality or a tracking gap, not a real loss."
2,3,176,Investigate drop,R3_DECLINING,0.9984,higher (n>=100 clicks),"Seasonality or a tracking gap, not a real loss."
3,4,117,Investigate drop,R3_DECLINING,0.9979,higher (n>=100 clicks),"Seasonality or a tracking gap, not a real loss."
4,5,68,Investigate drop,R3_DECLINING,0.9969,higher (n>=100 clicks),"Seasonality or a tracking gap, not a real loss."
5,6,201,Investigate drop,R3_DECLINING,0.9967,higher (n>=100 clicks),"Seasonality or a tracking gap, not a real loss."
6,7,126,Investigate drop,R3_DECLINING,0.9956,higher (n>=100 clicks),"Seasonality or a tracking gap, not a real loss."
7,8,269,Investigate drop,R3_DECLINING,0.9951,higher (n>=100 clicks),"Seasonality or a tracking gap, not a real loss."
8,9,291,Investigate drop,R3_DECLINING,0.9920,higher (n>=100 clicks),"Seasonality or a tracking gap, not a real loss."
9,10,55,Investigate drop,R3_DECLINING,0.9914,medium (30-99 clicks),"Seasonality or a tracking gap, not a real loss."


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [4]:
# Weak picks: top-20 rows with low evidence or only one thin signal
weak = top20[(top20[c["clicks"]] < 30) | (top20["rules_fired"] == 1) & (top20[c["impressions"]] < df[c["impressions"]].median())]
print(f"{len(weak)} of top 20 look weak (low clicks or thin single-rule evidence):")
display(weak[["rank", ID_COL, "reason_code", c["clicks"], c["impressions"], "score"]])

# Leakage check 1: no forbidden columns used by the score
used = set(COLS.values())
bad_used = [col for col in used if any(f in col.lower() for f in FORBIDDEN)]
print("Forbidden columns used in score:", bad_used or "none")

# Leakage check 2: forbidden columns present in the input but unused
present = [col for col in df.columns if any(f in col.lower() for f in FORBIDDEN)]
print("Forbidden-looking columns in table (must not feed score):", present or "none")

# Leakage check 3: comparison window is earlier, not later
print("prev window column is the EARLIER period:", c["prev_clicks"],
      "-> confirm against your data dictionary")

# Output sanity
assert ranked["rank"].is_unique and ranked["score"].is_monotonic_decreasing
assert ranked["reason_code"].isin(REASONS).all()
print("Checks passed (observed, directional, decision-support only).")

0 of top 20 look weak (low clicks or thin single-rule evidence):


,rank,item_id,reason_code,clicks,impressions,score


Forbidden columns used in score: none
Forbidden-looking columns in table (must not feed score): none
prev window column is the EARLIER period: prev_clicks -> confirm against your data dictionary
Checks passed (observed, directional, decision-support only).


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.